In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge

In [ ]:
data = {
    'Outlook': ['sunny', 'sunny', 'overcast', 'rain', 'rain', 'rain', 'overcast', 'sunny', 'sunny', 
                'rain', 'sunny', 'overcast', 'overcast', 'rain', 'sunny', 'overcast', 'rain', 'sunny', 
                'sunny', 'rain', 'overcast', 'rain', 'sunny', 'overcast', 'sunny', 'overcast', 'rain', 'overcast'],
    'Temperature': [85, 80, 83, 70, 68, 65, 64, 72, 69, 75, 75, 72, 81, 71, 81, 74, 76, 78, 82, 
                   67, 85, 73, 88, 77, 79, 80, 66, 84],
    'Humidity': [85, 90, 78, 96, 80, 70, 65, 95, 70, 80, 70, 90, 75, 80, 88, 92, 85, 75, 92, 
                 90, 85, 88, 65, 70, 60, 95, 70, 78],
    'Wind': [False, True, False, False, False, True, True, False, False, False, True, True, False, 
             True, True, False, False, True, False, True, True, False, True, False, False, True, False, False],
    'Num_Players': [52, 39, 43, 37, 28, 19, 43, 47, 56, 33, 49, 23, 42, 13, 33, 29, 25, 51, 41, 
                    14, 34, 29, 49, 36, 57, 21, 23, 41]
}

In [3]:
df = pd.get_dummies(pd.DataFrame(data), columns=['Outlook'], prefix='', prefix_sep='', dtype=int)
df

,Temperature,Humidity,Wind,Num_Players,overcast,rain,sunny
0,85,85,False,52,0,0,1
1,80,90,True,39,0,0,1
2,83,78,False,43,1,0,0
3,70,96,False,37,0,1,0
4,68,80,False,28,0,1,0
5,65,70,True,19,0,1,0
6,64,65,True,43,1,0,0
7,72,95,False,47,0,0,1
8,69,70,False,56,0,0,1
9,75,80,False,33,0,1,0


In [4]:
df['Wind'] = df['Wind'].astype(int)


In [5]:
df

,Temperature,Humidity,Wind,Num_Players,overcast,rain,sunny
0,85,85,0,52,0,0,1
1,80,90,1,39,0,0,1
2,83,78,0,43,1,0,0
3,70,96,0,37,0,1,0
4,68,80,0,28,0,1,0
5,65,70,1,19,0,1,0
6,64,65,1,43,1,0,0
7,72,95,0,47,0,0,1
8,69,70,0,56,0,0,1
9,75,80,0,33,0,1,0


In [6]:
df = df[['sunny','overcast','rain','Temperature','Humidity','Wind','Num_Players']]
df

,sunny,overcast,rain,Temperature,Humidity,Wind,Num_Players
0,1,0,0,85,85,0,52
1,1,0,0,80,90,1,39
2,0,1,0,83,78,0,43
3,0,0,1,70,96,0,37
4,0,0,1,68,80,0,28
5,0,0,1,65,70,1,19
6,0,1,0,64,65,1,43
7,1,0,0,72,95,0,47
8,1,0,0,69,70,0,56
9,0,0,1,75,80,0,33


In [7]:
X, y = df.drop(columns='Num_Players'), df['Num_Players']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)


In [8]:
# Scale numerical features
numerical_cols = ['Temperature', 'Humidity']
ct = ColumnTransformer([('scaler', StandardScaler(), numerical_cols)], remainder='passthrough')


In [9]:
# Transform data
X_train_scaled = pd.DataFrame(
    ct.fit_transform(X_train),
    columns=numerical_cols + [col for col in X_train.columns if col not in numerical_cols],
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    ct.transform(X_test),
    columns=X_train_scaled.columns,
    index=X_test.index
)

In [10]:
# Initialize and train the model
model_OLS = LinearRegression() # Option 1: OLS Regression
model_Ridge = Ridge(alpha=0.1)  # Option 2: Ridge Regression (alpha is the regularization strength, equivalent to λ)


In [12]:
model_OLS.fit(X_train_scaled,y_train)


LinearRegression()

In [13]:
model_Ridge.fit(X_train_scaled,y_train)

Ridge(alpha=0.1)

In [14]:
# Make predictions
y_pred_OLS = model_OLS.predict(X_test_scaled)
y_pred_Ridge = model_Ridge.predict(X_test_scaled)

# Calculate and print RMSE
rmse_OLS = root_mean_squared_error(y_test, y_pred_OLS)
rmse_Ridge = root_mean_squared_error(y_test, y_pred_Ridge)
print(f"RMSE for OLS: {rmse_OLS:.4f}")
print(f"RMSE for Ridge: {rmse_Ridge:.4f}")



RMSE for OLS: 7.0260
RMSE for Ridge: 6.8940


In [ ]:
# Additional information about the model
print("OLS Model Coefficients:")
print(f"Intercept    : {model_OLS.intercept_:.2f}")
for feature, coef in zip(X_train_scaled.columns, model_OLS.coef_):
    print(f"{feature:13}: {coef:.2f}")

nOLSModel Coefficients:
Intercept    : 43.25
Temperature  : -1.07
Humidity     : -2.85
sunny        : 11.71
overcast     : 0.50
rain         : -12.22
Wind         : -13.49


In [ ]:
# Additional information about the model
print("nOLSModel Coefficients:")
print(f"Intercept    : {model_Ridge.intercept_:.2f}")
for feature, coef in zip(X_train_scaled.columns, model_Ridge.coef_):
    print(f"{feature:13}: {coef:.2f}")